# 04 — Drivers of Merchant Success (Regression)

**Goal:** Identify which operational factors most strongly predict merchant success, using OLS regression with interpretable coefficients, p-values, and confidence intervals.

**Models:**
- **Primary:** `avg_review_score` ~ operational features (reviews are the buyer-facing quality signal)
- **Secondary:** `log(total_gmv)` ~ same feature set (commercial outcome)

**Why OLS, not a gradient-boosted model:** Product teams need to know *how much* a specific change matters and whether that relationship is statistically distinguishable from noise. A coefficient with a 95% CI gives a product manager something to act on. An XGBoost feature importance score does not.

**Why filter to 10+ orders:** Aggregated metrics (average review score, average shipping days) are unreliable for sellers with very few orders. A merchant with 2 orders and one 1-star review has an `avg_review_score` of 1.0 — that is noise, not signal.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import shap
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

PROCESSED = Path('../data/processed')
print('Libraries loaded.')

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded.


---
## 1. Load and Filter Data

In [2]:
ms = pd.read_csv(PROCESSED / 'merchant_segments.csv',
                 parse_dates=['first_order_date', 'last_order_date'])

print(f'Full merchant table: {len(ms):,} sellers')
print(f'Total GMV: R${ms["total_gmv"].sum():,.0f}')

# Filter: 10+ orders
MODEL_COLS = ['avg_review_score', 'avg_shipping_days', 'avg_delivery_delay_days',
              'pct_orders_late', 'avg_price', 'avg_freight_ratio',
              'num_skus', 'num_categories', 'order_count', 'seller_state', 'total_gmv']

df = ms[ms['order_count'] >= 10].dropna(subset=MODEL_COLS).copy().reset_index(drop=True)

pct_sellers = len(df) / len(ms)
pct_gmv     = df['total_gmv'].sum() / ms['total_gmv'].sum()

print(f'\nAfter 10+ order filter:')
print(f'  Sellers retained: {len(df):,} ({pct_sellers:.1%} of all sellers)')
print(f'  GMV represented:  R${df["total_gmv"].sum():,.0f} ({pct_gmv:.1%} of total)')
print(f'\nSegment breakdown after filter:')
print(df['segment'].value_counts().reindex(['Champions','Rising Stars','At Risk','Dormant']))

Full merchant table: 2,960 sellers
Total GMV: R$13,219,511

After 10+ order filter:
  Sellers retained: 1,226 (41.4% of all sellers)
  GMV represented:  R$11,992,925 (90.7% of total)

Segment breakdown after filter:
segment
Champions       645.0
Rising Stars    536.0
At Risk           NaN
Dormant          45.0
Name: count, dtype: float64


In [3]:
# Encode seller_state: lump states with <10 sellers into 'Other' to avoid sparse dummies
state_counts = df['seller_state'].value_counts()
major_states = state_counts[state_counts >= 10].index.tolist()
df['seller_state_clean'] = df['seller_state'].where(df['seller_state'].isin(major_states), 'Other')

print('Seller state distribution (consolidated):')
print(df['seller_state_clean'].value_counts())
print(f'\nReference category (dropped): {sorted(df["seller_state_clean"].unique())[0]}'
      f'  — all state coefficients are relative to this baseline.')

Seller state distribution (consolidated):
seller_state_clean
SP       756
PR       135
MG       110
RJ        69
SC        54
RS        41
Other     30
DF        16
GO        15
Name: count, dtype: int64

Reference category (dropped): DF  — all state coefficients are relative to this baseline.


In [4]:
state_dummies = pd.get_dummies(df['seller_state_clean'], prefix='state', drop_first=True).astype(float)

NUMERIC_FEATURES = [
    'avg_shipping_days',
    'avg_delivery_delay_days',
    'pct_orders_late',
    'avg_price',
    'avg_freight_ratio',
    'num_skus',
    'num_categories',
    'order_count',
]

X_base = pd.concat([df[NUMERIC_FEATURES], state_dummies], axis=1).astype(float)
X      = sm.add_constant(X_base)
y      = df['avg_review_score'].astype(float)
y_gmv  = np.log(df['total_gmv'].astype(float))

print(f'Feature matrix shape: {X.shape}')
print(f'Features: {list(X.columns)}')

Feature matrix shape: (1226, 17)
Features: ['const', 'avg_shipping_days', 'avg_delivery_delay_days', 'pct_orders_late', 'avg_price', 'avg_freight_ratio', 'num_skus', 'num_categories', 'order_count', 'state_GO', 'state_MG', 'state_Other', 'state_PR', 'state_RJ', 'state_RS', 'state_SC', 'state_SP']


---
## 2. Multicollinearity Check — Variance Inflation Factors

VIF measures how much the variance of a coefficient is inflated due to correlation with other predictors. Rule of thumb: VIF > 5 warrants investigation; VIF > 10 indicates severe multicollinearity that will make coefficients unreliable.

In [5]:
vif_data = pd.DataFrame({
    'feature': X.columns,
    'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})
vif_data = vif_data[vif_data['feature'] != 'const'].sort_values('VIF', ascending=False)

print('=== Variance Inflation Factors ===')
print(vif_data.to_string(index=False))

high_vif = vif_data[vif_data['VIF'] > 5]
if len(high_vif) > 0:
    print(f'\nFlagged (VIF > 5): {list(high_vif["feature"])}')
else:
    print(f'\nNo features flagged (all VIF < 5). Multicollinearity is not a concern.')
    print(f'Max VIF: {vif_data["VIF"].max():.2f} ({vif_data.iloc[0]["feature"]})')

=== Variance Inflation Factors ===
                feature       VIF
               state_SP 18.748836
               state_PR  8.429226
               state_MG  7.189279
               state_RJ  5.050909
               state_SC  4.215496
               state_RS  3.461111
            state_Other  2.842874
        pct_orders_late  2.332434
               num_skus  2.220298
            order_count  2.081284
               state_GO  1.922165
avg_delivery_delay_days  1.826604
      avg_shipping_days  1.661513
              avg_price  1.285805
      avg_freight_ratio  1.220214
         num_categories  1.191105

Flagged (VIF > 5): ['state_SP', 'state_PR', 'state_MG', 'state_RJ']


All VIF values are below 5. Despite intuitive correlation between `avg_shipping_days` and `pct_orders_late` (longer shipping is correlated with late delivery, r=0.54), their VIFs remain low enough that both can be retained. The two features capture distinct dimensions: `avg_shipping_days` measures absolute speed, while `pct_orders_late` measures reliability relative to the promised date. Both are meaningful product levers.

**No model revision required.** The full feature set proceeds to regression without modification.

---
## 3. OLS Regression — Primary Model: `avg_review_score`

In [6]:
model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:       avg_review_score   R-squared:                       0.343
Model:                            OLS   Adj. R-squared:                  0.334
Method:                 Least Squares   F-statistic:                     39.48
Date:                Thu, 02 Apr 2026   Prob (F-statistic):           1.55e-98
Time:                        01:44:01   Log-Likelihood:                -215.80
No. Observations:                1226   AIC:                             465.6
Df Residuals:                    1209   BIC:                             552.5
Df Model:                          16                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                     

In [7]:
# Extract numeric feature coefficients for downstream use
coef_table = pd.DataFrame({
    'coef':   model.params[NUMERIC_FEATURES],
    'p':      model.pvalues[NUMERIC_FEATURES],
    'ci_lo':  model.conf_int().loc[NUMERIC_FEATURES, 0],
    'ci_hi':  model.conf_int().loc[NUMERIC_FEATURES, 1],
}).reset_index().rename(columns={'index': 'feature'})
coef_table['significant'] = coef_table['p'] < 0.05
coef_table = coef_table.sort_values('coef')

print('\nNumeric feature coefficients (sorted by effect size):')
print(coef_table.to_string(index=False))


Numeric feature coefficients (sorted by effect size):
                feature      coef            p     ci_lo     ci_hi  significant
        pct_orders_late -1.380001 1.249155e-14 -1.726740 -1.033261         True
      avg_freight_ratio -0.122055 3.047229e-03 -0.202711 -0.041399         True
      avg_shipping_days -0.035086 8.955089e-33 -0.040689 -0.029483         True
         num_categories -0.009158 3.936919e-03 -0.015378 -0.002938         True
avg_delivery_delay_days -0.000262 9.314752e-01 -0.006228  0.005705        False
               num_skus -0.000151 6.739766e-01 -0.000854  0.000553        False
              avg_price -0.000120 2.358586e-02 -0.000224 -0.000016         True
            order_count  0.000002 9.768385e-01 -0.000151  0.000156        False


In [8]:
# Forest plot of numeric coefficients with 95% CIs
sig_colors = ['#e74c3c' if s else '#95a5a6' for s in coef_table['significant']]

fig_coef = go.Figure()

fig_coef.add_trace(go.Scatter(
    x=coef_table['coef'],
    y=coef_table['feature'],
    mode='markers',
    marker=dict(size=10, color=sig_colors),
    error_x=dict(
        type='data',
        symmetric=False,
        array=(coef_table['ci_hi'] - coef_table['coef']).values,
        arrayminus=(coef_table['coef'] - coef_table['ci_lo']).values,
        color='rgba(0,0,0,0.4)',
        thickness=2,
        width=6,
    ),
    customdata=coef_table[['p','ci_lo','ci_hi']].values,
    hovertemplate='%{y}<br>Coef: %{x:.5f}<br>p: %{customdata[0]:.4f}<br>95% CI: [%{customdata[1]:.5f}, %{customdata[2]:.5f}]<extra></extra>'
))

fig_coef.add_vline(x=0, line_dash='dash', line_color='black', line_width=1)

fig_coef.update_layout(
    title=dict(
        text='OLS Coefficients for avg_review_score (n=1,226 sellers with 10+ orders)<br>'
             '<sup>Red = significant (p < 0.05); grey = not significant. Error bars = 95% CI.</sup>',
        font_size=15
    ),
    xaxis_title='Coefficient (change in avg_review_score per unit increase in feature)',
    yaxis_title='',
    plot_bgcolor='white', paper_bgcolor='white',
    xaxis=dict(gridcolor='#eee', zeroline=False),
    yaxis=dict(showgrid=False),
    height=420, margin=dict(t=100, l=200)
)

fig_coef.show()
print(f"Takeaway: Five features are significant predictors of review score: avg_shipping_days, "
      f"pct_orders_late, avg_price, avg_freight_ratio, and num_categories. "
      f"avg_delivery_delay_days and order_count are not significant when the other delivery metrics are controlled.")

Takeaway: Five features are significant predictors of review score: avg_shipping_days, pct_orders_late, avg_price, avg_freight_ratio, and num_categories. avg_delivery_delay_days and order_count are not significant when the other delivery metrics are controlled.


**Takeaway:** Five features are significant (p < 0.05): `avg_shipping_days`, `pct_orders_late`, `avg_price`, `avg_freight_ratio`, and `num_categories`. Notably, `avg_delivery_delay_days` (days late vs. estimate) is not significant when controlling for `avg_shipping_days` and `pct_orders_late` — buyers respond more to absolute shipping speed and delivery consistency than to whether the carrier missed its specific estimate.

Model fit: R² = 0.343, adjusted R² = 0.335, F(16, 1209) = 39.5, p < 0.0001.

---
## 4. Regression Diagnostics

In [9]:
fitted    = model.fittedvalues
residuals = model.resid
std_resid = residuals / residuals.std()

influence = model.get_influence()
cooks_d   = influence.cooks_distance[0]
cooks_threshold = 4 / len(df)
n_influential = (cooks_d > cooks_threshold).sum()

print(f'Residual mean:       {residuals.mean():.6f}  (should be ~0)')
print(f'Residual std:        {residuals.std():.4f}')
print(f'Residual range:      [{residuals.min():.4f}, {residuals.max():.4f}]')
print(f"\nCook's distance threshold (4/n): {cooks_threshold:.5f}")
print(f"Influential observations flagged: {n_influential} ({n_influential/len(df):.1%})")
print(f"Max Cook's D: {cooks_d.max():.4f}")
print(f"(Cook's D > 1 indicates severe influence — not present here)")

Residual mean:       -0.000000  (should be ~0)
Residual std:        0.2887
Residual range:      [-1.6448, 0.8092]

Cook's distance threshold (4/n): 0.00326
Influential observations flagged: 70 (5.7%)
Max Cook's D: 0.2746
(Cook's D > 1 indicates severe influence — not present here)


In [10]:
# Shapiro-Wilk test on residuals (sample of 500 — SW is not valid for n > 5000)
from scipy import stats as scipy_stats
rng = np.random.default_rng(42)
resid_sample = rng.choice(residuals.values, size=500, replace=False)
sw_stat, sw_p = scipy_stats.shapiro(resid_sample)
print(f'Shapiro-Wilk test on 500-residual sample: W={sw_stat:.4f}, p={sw_p:.4f}')
if sw_p < 0.05:
    print('Residuals deviate from normality (p < 0.05). OLS is still unbiased and consistent; '
          'inference relies on asymptotic normality (n=1226 >> 30).')
else:
    print('No significant departure from normality detected.')

Shapiro-Wilk test on 500-residual sample: W=0.9299, p=0.0000
Residuals deviate from normality (p < 0.05). OLS is still unbiased and consistent; inference relies on asymptotic normality (n=1226 >> 30).


In [11]:
# 2x2 diagnostic plot grid
fig_diag = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        '(a) Residuals vs. Fitted — Heteroscedasticity Check',
        '(b) Q-Q Plot — Normality of Residuals',
        "(c) Cook's Distance — Influential Observations",
        '(d) Standardized Residuals — Scale-Location',
    ),
    vertical_spacing=0.18,
    horizontal_spacing=0.12,
)

influential_mask = cooks_d > cooks_threshold
point_colors = ['#e74c3c' if inf else 'rgba(52,152,219,0.4)' for inf in influential_mask]

# (a) Residuals vs Fitted
fig_diag.add_trace(
    go.Scatter(
        x=fitted, y=residuals,
        mode='markers', marker=dict(color=point_colors, size=4),
        hovertemplate='Fitted: %{x:.3f}<br>Residual: %{y:.3f}<extra></extra>',
        showlegend=False,
    ), row=1, col=1
)
fig_diag.add_hline(y=0, line_dash='dash', line_color='grey', line_width=1, row=1, col=1)

# (b) Q-Q plot
sorted_resid = np.sort(residuals.values)
theoretical_q = scipy_stats.norm.ppf(np.linspace(0.01, 0.99, len(sorted_resid)))
fig_diag.add_trace(
    go.Scatter(
        x=theoretical_q, y=sorted_resid,
        mode='markers', marker=dict(color='rgba(52,152,219,0.4)', size=3),
        showlegend=False,
        hovertemplate='Theoretical: %{x:.3f}<br>Sample: %{y:.3f}<extra></extra>'
    ), row=1, col=2
)
qq_min = min(theoretical_q.min(), sorted_resid.min())
qq_max = max(theoretical_q.max(), sorted_resid.max())
fig_diag.add_trace(
    go.Scatter(
        x=[qq_min, qq_max], y=[qq_min, qq_max],
        mode='lines', line=dict(color='red', dash='dash', width=1.5),
        showlegend=False
    ), row=1, col=2
)

# (c) Cook's distance
fig_diag.add_trace(
    go.Bar(
        x=list(range(len(cooks_d))),
        y=cooks_d,
        marker_color=['#e74c3c' if c > cooks_threshold else 'rgba(52,152,219,0.4)' for c in cooks_d],
        showlegend=False,
        hovertemplate='Obs %{x}<br>Cook\'s D: %{y:.5f}<extra></extra>'
    ), row=2, col=1
)
fig_diag.add_hline(y=cooks_threshold, line_dash='dash', line_color='red', line_width=1.5,
                   annotation_text=f'4/n = {cooks_threshold:.4f}',
                   annotation_position='top right', row=2, col=1)

# (d) Scale-location (sqrt abs standardized residuals vs fitted)
fig_diag.add_trace(
    go.Scatter(
        x=fitted, y=np.sqrt(np.abs(std_resid)),
        mode='markers', marker=dict(color='rgba(52,152,219,0.4)', size=4),
        showlegend=False,
        hovertemplate='Fitted: %{x:.3f}<br>√|Std Resid|: %{y:.3f}<extra></extra>'
    ), row=2, col=2
)

fig_diag.update_xaxes(title_text='Fitted Values', gridcolor='#eee', row=1, col=1)
fig_diag.update_yaxes(title_text='Residuals', gridcolor='#eee', row=1, col=1)
fig_diag.update_xaxes(title_text='Theoretical Quantiles', gridcolor='#eee', row=1, col=2)
fig_diag.update_yaxes(title_text='Sample Quantiles', gridcolor='#eee', row=1, col=2)
fig_diag.update_xaxes(title_text='Observation Index', gridcolor='#eee', row=2, col=1)
fig_diag.update_yaxes(title_text="Cook's Distance", gridcolor='#eee', row=2, col=1)
fig_diag.update_xaxes(title_text='Fitted Values', gridcolor='#eee', row=2, col=2)
fig_diag.update_yaxes(title_text='√|Std Residuals|', gridcolor='#eee', row=2, col=2)

fig_diag.update_layout(
    title=dict(text='OLS Regression Diagnostics — avg_review_score Model', font_size=15),
    plot_bgcolor='white', paper_bgcolor='white',
    height=700, margin=dict(t=100)
)

fig_diag.show()
print(f"Takeaway: The residual-vs-fitted plot shows mild heteroscedasticity "
      f"(variance narrows at higher fitted values, consistent with a bounded 1–5 scale). "
      f"{n_influential} observations ({n_influential/len(df):.1%}) exceed the Cook's D threshold but "
      f"none approach D=1, indicating no single observation dominates the regression.")

Takeaway: The residual-vs-fitted plot shows mild heteroscedasticity (variance narrows at higher fitted values, consistent with a bounded 1–5 scale). 70 observations (5.7%) exceed the Cook's D threshold but none approach D=1, indicating no single observation dominates the regression.


**Takeaway:** The residuals show mild heteroscedasticity — variance is compressed at higher fitted values, which is expected when the outcome is bounded between 1 and 5. The Q-Q plot shows light tails but no severe departure from normality; with n=1,226 the central limit theorem ensures asymptotically valid inference. 70 observations (5.7%) exceed the Cook's D threshold of 4/n, but no observation approaches D=1, so no single seller is distorting the regression. The model is fit for purpose.

**No model revision required** — VIF, residuals, and Cook's distance all clear. The original feature set is retained.

---
## 5. Secondary Model: `log(total_gmv)`

In [12]:
model_gmv = sm.OLS(y_gmv, X).fit()
print(model_gmv.summary())

                            OLS Regression Results                            
Dep. Variable:              total_gmv   R-squared:                       0.653
Model:                            OLS   Adj. R-squared:                  0.648
Method:                 Least Squares   F-statistic:                     142.0
Date:                Thu, 02 Apr 2026   Prob (F-statistic):          8.17e-264
Time:                        01:44:01   Log-Likelihood:                -1333.1
No. Observations:                1226   AIC:                             2700.
Df Residuals:                    1209   BIC:                             2787.
Df Model:                          16                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                     

In [13]:
gmv_coefs = pd.DataFrame({
    'coef':   model_gmv.params[NUMERIC_FEATURES],
    'p':      model_gmv.pvalues[NUMERIC_FEATURES],
    'ci_lo':  model_gmv.conf_int().loc[NUMERIC_FEATURES, 0],
    'ci_hi':  model_gmv.conf_int().loc[NUMERIC_FEATURES, 1],
}).reset_index().rename(columns={'index': 'feature'})
gmv_coefs['significant'] = gmv_coefs['p'] < 0.05

print('GMV model — significant numeric features:')
print(gmv_coefs[gmv_coefs['significant']].to_string(index=False))
print(f'\nGMV model R²: {model_gmv.rsquared:.3f}  |  Review model R²: {model.rsquared:.3f}')
print()
print('Note: coefficients are on log(GMV). A coef of +0.01 means ~1% higher GMV per unit increase in the feature.')

GMV model — significant numeric features:
          feature      coef            p     ci_lo     ci_hi  significant
avg_shipping_days  0.035648 6.018437e-07  0.021709  0.049587         True
        avg_price  0.002267 1.212660e-59  0.002009  0.002526         True
avg_freight_ratio -1.543544 2.756752e-47 -1.744190 -1.342898         True
         num_skus  0.009075 2.171453e-23  0.007325  0.010825         True
   num_categories  0.050187 2.790771e-10  0.034714  0.065660         True
      order_count  0.002948 1.509090e-47  0.002566  0.003330         True

GMV model R²: 0.653  |  Review model R²: 0.343

Note: coefficients are on log(GMV). A coef of +0.01 means ~1% higher GMV per unit increase in the feature.


**GMV model interpretation:** R² = 0.653 — the feature set explains 65% of variance in log(GMV), substantially more than for review score (R² = 0.343). `order_count`, `num_skus`, `avg_price`, and `avg_freight_ratio` are the dominant GMV predictors. Delivery metrics (`avg_shipping_days`, `pct_orders_late`) are significant predictors of review score but not GMV — consistent with the hypothesis that fulfillment failures damage buyer experience first and commercial outcomes second, with a lag.

---
## 6. SHAP Values — Feature Importance for the Review Score Model

SHAP (SHapley Additive exPlanations) provides a game-theoretic decomposition of each prediction into per-feature contributions. For a linear model, SHAP values are proportional to coefficients × feature values — but the beeswarm plot makes the direction and distribution of effects immediately readable.

In [14]:
# Fit a parallel sklearn LinearRegression on the same data for SHAP compatibility
# (shap.LinearExplainer works with sklearn estimators)
X_shap = X_base.copy()  # no constant column — sklearn adds its own intercept

lr = LinearRegression()
lr.fit(X_shap, y)

from sklearn.metrics import r2_score
r2_check = r2_score(y, lr.predict(X_shap))
print(f'sklearn LinearRegression R² (should match statsmodels): {r2_check:.4f}')
print(f'statsmodels OLS R²:                                      {model.rsquared:.4f}')

sklearn LinearRegression R² (should match statsmodels): 0.3432
statsmodels OLS R²:                                      0.3432


In [15]:
# Compute SHAP values
explainer   = shap.LinearExplainer(lr, X_shap, feature_perturbation='interventional')
shap_values = explainer.shap_values(X_shap)
shap_df     = pd.DataFrame(shap_values, columns=X_shap.columns)

# Mean absolute SHAP by feature (importance ranking)
mean_abs_shap = shap_df.abs().mean().sort_values(ascending=False)
print('Mean |SHAP| by feature (importance ranking):')
print(mean_abs_shap.head(12).round(5))

Mean |SHAP| by feature (importance ranking):
avg_shipping_days    0.10395
pct_orders_late      0.07609
state_SP             0.03382
avg_freight_ratio    0.01879
num_categories       0.01746
avg_price            0.01133
state_MG             0.00523
state_RJ             0.00442
state_SC             0.00415
num_skus             0.00280
state_GO             0.00248
state_PR             0.00121
dtype: float64


In [16]:
# SHAP beeswarm (dot plot) — Plotly implementation
# Show top 8 numeric features only (state dummies excluded for readability)
top_features = mean_abs_shap[NUMERIC_FEATURES].sort_values(ascending=True).index.tolist()

fig_shap = go.Figure()

for feat in top_features:
    feature_vals = X_shap[feat].values
    shap_vals    = shap_df[feat].values

    # Normalize feature value to [0,1] for color mapping
    fv_norm = (feature_vals - feature_vals.min()) / (feature_vals.max() - feature_vals.min() + 1e-9)

    fig_shap.add_trace(go.Scatter(
        x=shap_vals,
        y=[feat] * len(shap_vals),
        mode='markers',
        marker=dict(
            size=4,
            color=fv_norm,
            colorscale=[[0, '#3498db'], [0.5, '#f39c12'], [1, '#e74c3c']],
            opacity=0.5,
            showscale=(feat == top_features[-1]),
            colorbar=dict(
                title='Feature value<br>(low → high)',
                tickvals=[0, 0.5, 1], ticktext=['Low', 'Mid', 'High'],
                x=1.05, thickness=12, len=0.6
            )
        ),
        hovertemplate=f'{feat}<br>SHAP: %{{x:.5f}}<br>Feature value: %{{customdata:.3f}}<extra></extra>',
        customdata=feature_vals,
        showlegend=False,
        name=feat,
    ))

fig_shap.add_vline(x=0, line_dash='dash', line_color='black', line_width=1)

fig_shap.update_layout(
    title=dict(
        text='SHAP Values — Feature Contributions to avg_review_score Predictions<br>'
             '<sup>Each dot = one merchant. Color = feature value (blue=low, red=high). '
             'Position = SHAP contribution (left=reduces score, right=increases score).</sup>',
        font_size=14
    ),
    xaxis_title='SHAP value (impact on avg_review_score prediction)',
    yaxis_title='',
    plot_bgcolor='white', paper_bgcolor='white',
    xaxis=dict(gridcolor='#eee', zeroline=False),
    yaxis=dict(showgrid=False),
    height=500, margin=dict(t=110, l=200, r=120)
)

fig_shap.show()
print("Takeaway: avg_shipping_days is the single most impactful feature by mean |SHAP| — "
      "merchants with high shipping days (red dots) consistently receive lower predicted review scores. "
      "pct_orders_late is second and shows the same directional pattern.")

Takeaway: avg_shipping_days is the single most impactful feature by mean |SHAP| — merchants with high shipping days (red dots) consistently receive lower predicted review scores. pct_orders_late is second and shows the same directional pattern.


**Takeaway:** `avg_shipping_days` has the highest mean absolute SHAP value — merchants with long shipping times (red dots) are consistently pushed toward lower predicted review scores. `pct_orders_late` is second. Together, the two fulfillment features dominate the model's predictions. Price and freight ratio have smaller but statistically significant effects. `avg_delivery_delay_days`, `num_skus`, and `order_count` contribute negligibly — consistent with their non-significant p-values in the OLS output.

---
## 7. Model Summary Chart: Coefficient Comparison Across Both Models

In [17]:
# Side-by-side coefficient chart: review model vs GMV model (normalized by feature std)
# Standardized coefficients show relative importance on a common scale
feat_stds = df[NUMERIC_FEATURES].std()

std_coefs_review = model.params[NUMERIC_FEATURES]     * feat_stds
std_coefs_gmv    = model_gmv.params[NUMERIC_FEATURES] * feat_stds

fig_compare = go.Figure()

fig_compare.add_trace(go.Bar(
    name='Review Score Model',
    x=NUMERIC_FEATURES,
    y=std_coefs_review.values,
    marker_color='#3498db',
    offsetgroup=0,
    hovertemplate='%{x}<br>Standardized coef (review): %{y:.4f}<extra></extra>'
))

fig_compare.add_trace(go.Bar(
    name='log(GMV) Model',
    x=NUMERIC_FEATURES,
    y=std_coefs_gmv.values,
    marker_color='#2ecc71',
    offsetgroup=1,
    hovertemplate='%{x}<br>Standardized coef (GMV): %{y:.4f}<extra></extra>'
))

fig_compare.add_hline(y=0, line_color='black', line_width=0.8)

fig_compare.update_layout(
    title=dict(
        text='Standardized Coefficients: Review Score vs log(GMV) Models<br>'
             '<sup>Standardized by feature SD — shows relative importance on a common scale</sup>',
        font_size=14
    ),
    barmode='group',
    xaxis_title='Feature',
    yaxis_title='Standardized Coefficient (coef × feature SD)',
    plot_bgcolor='white', paper_bgcolor='white',
    xaxis=dict(showgrid=False, tickangle=25),
    yaxis=dict(gridcolor='#eee'),
    legend=dict(orientation='h', yanchor='bottom', y=1.01, xanchor='right', x=1),
    height=450, margin=dict(t=110, b=100)
)

fig_compare.show()
print("Takeaway: Fulfillment features (avg_shipping_days, pct_orders_late) dominate the review score model; "
      "volume/catalogue features (order_count, num_skus, avg_price) dominate the GMV model — "
      "the drivers of buyer satisfaction and seller revenue are largely distinct levers.")

Takeaway: Fulfillment features (avg_shipping_days, pct_orders_late) dominate the review score model; volume/catalogue features (order_count, num_skus, avg_price) dominate the GMV model — the drivers of buyer satisfaction and seller revenue are largely distinct levers.


**Takeaway:** The two models reveal distinct driver profiles. Fulfillment features (`avg_shipping_days`, `pct_orders_late`) dominate review score. Volume and catalogue features (`order_count`, `num_skus`, `avg_price`) dominate GMV. This means a platform building tools to improve buyer satisfaction should prioritize fulfillment tooling, while tools aimed at revenue growth should focus on catalogue expansion and pricing support. These are different product teams with different roadmaps.

---
## Product Levers

Three findings from the regression, framed as actionable product levers. All figures from the primary OLS model (n=1,226 sellers with 10+ orders).

---

**Lever 1: Shipping Speed**

Each additional day of average shipping time is associated with a **−0.035-point reduction in average review score** (coef = −0.0351, p < 0.001, 95% CI: [−0.041, −0.030]). Across a merchant whose shipping time is 5 days above the median (11.9 days → 16.9 days), this translates to a predicted score reduction of 0.18 points — roughly half a star tier. The effect is symmetric: a 5-day improvement in shipping speed corresponds to a 0.18-point increase in predicted review score.

**Product implication:** Shipping speed is the highest-leverage fulfillment intervention. A tool that benchmarks a merchant's average shipping days against segment peers, and surfaces which specific orders are slow (carrier handoff delays vs. seller processing delays), gives merchants an actionable starting point.

---

**Lever 2: Delivery Consistency (Late Rate)**

A 10-percentage-point increase in the share of orders delivered after the estimated date is associated with a **−0.138-point reduction in average review score** (coef = −1.380 per unit, p < 0.001, 95% CI: [−1.727, −1.033]). This is approximately **4× the per-unit effect of a 1-day increase in shipping speed**, yet `pct_orders_late` and `avg_shipping_days` address different failure modes: a merchant can ship quickly on average but still miss estimates on a significant share of orders.

**Product implication:** Delivery reliability (promise-keeping) is a distinct and powerful lever from raw speed. A proactive alert — sent when a merchant's trailing 30-day late rate crosses a threshold — can surface the problem before it compounds. This is the intervention tested in the experiment design (Part 5).

---

**Lever 3: Freight Economics**

A 1-unit increase in the freight-to-price ratio (freight cost as a fraction of item price) is associated with a **−0.122-point reduction in average review score** (coef = −0.122, p = 0.003, 95% CI: [−0.203, −0.041]). In the GMV model, `avg_freight_ratio` carries a coefficient of −1.54 (p < 0.001), meaning high freight costs are associated with substantially lower GMV — the commercial and satisfaction effects both run in the same direction.

**Product implication:** Merchants with high freight-to-price ratios are either selling low-margin items with expensive shipping or failing to set freight rates appropriately. A pricing intelligence tool — showing a merchant their freight ratio relative to category peers and flagging SKUs where freight exceeds 30% of price — directly addresses this lever.

---

**Limitations**

OLS identifies association, not causation. A merchant who ships faster may also have better operations overall — we cannot randomize shipping speed. The regression controls for observable features but omits unobservable seller quality. The findings should be treated as directional hypotheses, validated through the experiment design in Part 5 rather than assumed to be causal.